In [1]:
import numpy as np
import xtrack as xt
import xpart as xp
import matplotlib.pyplot as plt

In [2]:
# Load line
line = xt.Line.from_json('../../lattices/V105/reference_lattice/line_fccee_p_ring_LCC_105-0-0_z.json')
line.configure_radiation(model=None, model_beamstrahlung=None)

tw0 = line.twiss4d()
print(tw0.qx, tw0.qy)
print(tw0.dqx, tw0.dqy)
tt = line.get_table()

Loading line from dict: 100%|██████████| 21288/21288 [00:02<00:00, 7972.92it/s] 


Done loading line from dict.           
194.1666809035911 170.20000443507251
-0.19027745167934249 0.10179411049194853


### Inspect elements

In [8]:
# Dipoles
ttbend = tt.rows[tt.element_type=='RBend']
print('There are', len(ttbend), 'bends in the lattice.')

There are 2436 bends in the lattice.


In [11]:
# Quadrupoles
ttquad = tt.rows[tt.element_type=='Quadrupole']
mask = [quad for quad in ttquad.name if line.element_dict[quad].k1s==0 and abs(line.element_dict[quad].k1)>0]
ttquadno = ttquad.rows[np.isin(ttquad.name, mask)]
mask = [quad for quad in ttquad.name if abs(line.element_dict[quad].k1s)>0 and line.element_dict[quad].k1==0]
ttquadsk = ttquad.rows[np.isin(ttquad.name, mask)]
mask = [quad for quad in ttquad.name if abs(line.element_dict[quad].k1s)==0 and line.element_dict[quad].k1==0]
ttquadzero = ttquad.rows[np.isin(ttquad.name, mask)]
print('There are', len(ttquad), 'quads in the lattice, of which', len(ttquadno), 'are normal,', len(ttquadsk), 'are skew, and', len(ttquadzero), 'have zero strength.')

There are 2684 quads in the lattice, of which 2668 are normal, 0 are skew, and 16 have zero strength.


In [12]:
# Sextupoles
ttsext = tt.rows[tt.element_type=='Sextupole']
mask = [sext for sext in ttsext.name if line.element_dict[sext].k2s==0 and abs(line.element_dict[sext].k2)>0]
ttsextno = ttsext.rows[np.isin(ttsext.name, mask)]
mask = [sext for sext in ttsext.name if abs(line.element_dict[sext].k2s)>0 and line.element_dict[sext].k2==0]
ttsextsk = ttsext.rows[np.isin(ttsext.name, mask)]
mask = [sext for sext in ttsext.name if abs(line.element_dict[sext].k2s)==0 and line.element_dict[sext].k2==0]
ttsextzero = ttsext.rows[np.isin(ttsext.name, mask)]
print('There are', len(ttsext), 'sexts in the lattice, of which', len(ttsextno), 'are normal,', len(ttsextsk), 'are skew, and', len(ttsextzero), 'have zero strength.')

There are 1952 sexts in the lattice, of which 1950 are normal, 0 are skew, and 2 have zero strength.


In [14]:
# Monitors
ttbpm = tt.rows['bpm.*']
print('There are', len(ttbpm), 'BPMs in the lattice.')

There are 2219 BPMs in the lattice.


In [16]:
# Correctors
ttcorr = tt.rows['.*cor.*']
ttcorrh = tt.rows['.*hcor.*']
ttcorrv = tt.rows['.*vcor.*']
print('There are', len(ttcorr), 'correctors in the lattice, of which', len(ttcorrh), 'are horizontal and', len(ttcorrv), 'are vertical.')
line.element_refs[ttcorr.name[0]]._info()

There are 4518 correctors in the lattice, of which 2259 are horizontal and 2259 are vertical.
#  element_refs['vcor_qd0ar']._get_value()
   element_refs['vcor_qd0ar'] = Multipole(order=np.int64(0), inv_factorial_order=1, length=0, hxl=0, radiation_flag=np.int64(0), delta_taper=0, knl=[0.], ksl=[0.], knl_rel=[0.], ksl_rel=[0.], main_order=np.int32(0), main_is_skew=False, isthick=False, num_multipole_kicks=np.int64(0), model='adaptive', integrator='adaptive', shift_x=0, shift_y=0, shift_s=0, rot_s_rad=0, rot_x_rad=0, rot_y_rad=0, rot_s_rad_no_frame=0, rot_shift_anchor=0)

#  element_refs['vcor_qd0ar']._expr is None

#  element_refs['vcor_qd0ar'] does not influence any target



In [19]:
import sys
sys.path.append('../../helper_functions')
from helpers_for_imperfections_model import add_correctors

In [20]:
add_correctors(line, ttquadno.name, type='normal', order=1, switch_name='on_qno_corrector')
add_correctors(line, ttsextno.name, type='skew', order=1, switch_name='on_qsk_corrector')

In [ ]:
# Check that correctors have been added by looking at element_refs of a random quadrupole
line.element_refs[ttquadno.name[2]].knl[1]._info()

#  element_refs['qd0cr'].knl[1]._get_value()
   element_refs['qd0cr'].knl[1] = 0.0

#  element_refs['qd0cr'].knl[1]._expr
   element_refs['qd0cr'].knl[1] = (vars['knl1_qd0cr'] * vars['on_qno_corrector'])

#  element_refs['qd0cr'].knl[1]._expr._get_dependencies()
   vars['on_qno_corrector'] = 1
   vars['knl1_qd0cr'] = 0

#  element_refs['qd0cr'].knl[1] does not influence any target



In [23]:
# Check that correctors have been added by looking at element_refs of a random quadrupole
line.element_refs[ttsextno.name[2]].ksl[1]._info()

#  element_refs['sdy1r'].ksl[1]._get_value()
   element_refs['sdy1r'].ksl[1] = 0.0

#  element_refs['sdy1r'].ksl[1]._expr
   element_refs['sdy1r'].ksl[1] = (vars['ksl1_sdy1r'] * vars['on_qsk_corrector'])

#  element_refs['sdy1r'].ksl[1]._expr._get_dependencies()
   vars['on_qsk_corrector'] = 1
   vars['ksl1_sdy1r'] = 0

#  element_refs['sdy1r'].ksl[1] does not influence any target



In [ ]:
# Enable the correctors by setting the switches to 1
line.vars['on_qno_corrector'] = 1
line.vars['on_qsk_corrector'] = 1

In [25]:
vartable = line.vars.get_table()

In [26]:
qno_vars = vartable.rows['knl1_.*']
print(list(qno_vars.name))

['knl1_qd0ar', 'knl1_qd0br', 'knl1_qd0cr', 'knl1_qf1ar', 'knl1_qf1br', 'knl1_qf2r', 'knl1_qd3r', 'knl1_qd4r', 'knl1_qf5r', 'knl1_qd6r', 'knl1_qy1r', 'knl1_qy2r', 'knl1_qy3r', 'knl1_qy4r', 'knl1_qy3r.0', 'knl1_qy2r.0', 'knl1_qy1r.0', 'knl1_qd7r', 'knl1_qf8r', 'knl1_qd9r', 'knl1_qf10r', 'knl1_qd11r', 'knl1_qf12r', 'knl1_qx0r', 'knl1_qx1r', 'knl1_qx2r', 'knl1_qx1r.0', 'knl1_qx0r.0', 'knl1_qf13r', 'knl1_qd14r', 'knl1_qf15r', 'knl1_qd16r', 'knl1_qf17r', 'knl1_qd18r', 'knl1_qf19r', 'knl1_qd20r', 'knl1_qd10m', 'knl1_qf9m', 'knl1_qd8m', 'knl1_qd6m', 'knl1_qf5m', 'knl1_qd4m', 'knl1_qf3m', 'knl1_qd2m', 'knl1_qf1m', 'knl1_qd0m', 'knl1_qf0m', 'knl1_qd1a', 'knl1_qf2a', 'knl1_qd1a.0', 'knl1_qf3a', 'knl1_qd1a.1', 'knl1_qf2a.0', 'knl1_qd1a.2', 'knl1_qf2a.1', 'knl1_qd1am', 'knl1_qf2a.2', 'knl1_qd1a.3', 'knl1_qf2a.3', 'knl1_qd1a.4', 'knl1_qf3a.0', 'knl1_qd1a.5', 'knl1_qf2a.4', 'knl1_qd1a.6', 'knl1_qf2a.5', 'knl1_qd1am.1', 'knl1_qf2a.6', 'knl1_qd1a.7', 'knl1_qf2a.7', 'knl1_qd1a.8', 'knl1_qf3a.1', 'knl1_q

In [27]:
qsk_vars = vartable.rows['ksl1_.*']
print(list(qsk_vars.name))

['ksl1_sdm1r', 'ksl1_sdm1r.0', 'ksl1_sdy1r', 'ksl1_sdy1r.0', 'ksl1_sdy1r.1', 'ksl1_sdy1r.2', 'ksl1_sdy2r', 'ksl1_sdy2r.0', 'ksl1_sdy2r.1', 'ksl1_sdy2r.2', 'ksl1_sfm2r', 'ksl1_sfm2r.0', 'ksl1_sfx1r', 'ksl1_sfx1r.0', 'ksl1_sfx1r.1', 'ksl1_sfx1r.2', 'ksl1_sfx2r', 'ksl1_sfx2r.0', 'ksl1_sfx2r.1', 'ksl1_sfx2r.2', 'ksl1_scrabr', 'ksl1_scrabr.0', 'ksl1_scrabr.1', 'ksl1_scrabr.2', 'ksl1_sf3mr', 'ksl1_sf2bfr', 'ksl1_sd1bfr', 'ksl1_sf1bfr', 'ksl1_sd2bfr', 'ksl1_sd2afr', 'ksl1_sf1afr', 'ksl1_sd1afr', 'ksl1_sf2afr', 'ksl1_sf2a', 'ksl1_sd1a', 'ksl1_sf1a', 'ksl1_sd2a', 'ksl1_sd2a.0', 'ksl1_sf1a.0', 'ksl1_sd1a.0', 'ksl1_sf2a.0', 'ksl1_sf2a.1', 'ksl1_sd1a.1', 'ksl1_sf1a.1', 'ksl1_sd2a.1', 'ksl1_sd2a.2', 'ksl1_sf1a.2', 'ksl1_sd1a.2', 'ksl1_sf2a.2', 'ksl1_sf2a.3', 'ksl1_sd1a.3', 'ksl1_sf1a.3', 'ksl1_sd2a.3', 'ksl1_sd2a.4', 'ksl1_sf1a.4', 'ksl1_sd1a.4', 'ksl1_sf2a.4', 'ksl1_sf2a.5', 'ksl1_sd1a.5', 'ksl1_sf1a.5', 'ksl1_sd2a.5', 'ksl1_sd2a.6', 'ksl1_sf1a.6', 'ksl1_sd1a.6', 'ksl1_sf2a.6', 'ksl1_sf2a.7', 'ksl

In [28]:
# Save line with optics correctors installed
line.to_json('line_fccee_LCC_105-0-0_z_with_qno_qsk.json')